In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sns.set_context("poster")
sns.set_style("ticks")

In [ ]:
fi = pd.read_parquet("0.parquet")
spearman = pd.read_parquet("1.parquet")
weights = pd.read_parquet("2.parquet")
weights["AbsWeight"] = weights["Weight"].abs()

In [ ]:
id_cols = ["trainer.model_builder.param", "trainer.representations.layer", "cv"]

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman[spearman.split == "test"],
    x="trainer.representations.layer",
    y="mean",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title=None)
ax.set_xlabel("Layer")
ax.set_ylabel("Spearman")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig("../../../paper/figs/BERT_RC_CLS/layers/spearman.pdf", bbox_inches="tight")
plt.show()

# Training duration

In [ ]:
ax = sns.lineplot(
    weights,
    x="trainer.representations.layer",
    y="training_duration",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_xlabel("Layer")
ax.set_ylabel("Training duration (s)")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title=None)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/layers/training_duration.pdf", bbox_inches="tight"
)
plt.show()

# FI

In [ ]:
features_fi = (
    fi[(fi.split == "test")]
    .groupby(["Feature", "trainer.model_builder.param"])["mean"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features_weights = (
    weights.groupby(["Feature", "trainer.model_builder.param"])
    .AbsWeight.mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features = pd.concat([features_fi, features_weights])[["Feature"]].drop_duplicates()
features

In [ ]:
g = sns.relplot(
    fi[(fi.split == "test")].merge(features),
    kind="line",
    x="trainer.representations.layer",
    y="mean",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.set_xlabel("Layer")
    ax.set_ylabel("FI")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/layers/feature_importance.pdf", bbox_inches="tight"
)
plt.show()

# Weights

In [ ]:
g = sns.relplot(
    weights.merge(features),
    kind="line",
    x="trainer.representations.layer",
    y="AbsWeight",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.set_xlabel("Layer")
    ax.set_ylabel("AbsWeight")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig("../../../paper/figs/BERT_RC_CLS/layers/weights.pdf", bbox_inches="tight")
plt.show()

# Weighted $\tau$ by layer

In [ ]:
weightedtau = weights.merge(fi[fi.split == "test"], on=id_cols + ["Feature"])
weightedtau = (
    weightedtau.groupby(
        id_cols,
    )
    .apply(
        lambda x: stats.weightedtau(x.Weight.abs(), x["mean"].abs()).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    weightedtau,
    x="trainer.representations.layer",
    y="Weighted $\\tau$",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title=None)
ax.set_xlabel("Layer")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/layers/weighted_tau.pdf", bbox_inches="tight"
)
plt.show()